# AmazonQA-RAG — Standalone Smoke Test (Phase 1 + Phase 2)

> **Purpose:** Verify the full AmazonQA-RAG pipeline runs end-to-end on a Colab T4 in one notebook,
> with no external repo or file uploads beyond this `.ipynb`.

**What this notebook does (in order):**

1. Installs all dependencies in ONE atomic pip transaction (no version drift)
2. Restarts the kernel so freshly-installed NumPy 1.x is loaded into memory
3. Inlines `configs/config.py`, `src/finetune.py`, `src/llm_loader.py`, `src/rag_pipeline.py` as code cells
4. Runs a **fast** Phase 1 fine-tune (500 samples, 50 steps - ~5 min)
5. Builds a Phase 2 FAISS index over a 1000-doc corpus
6. Compares Base LLM vs RAG+Base vs RAG+Fine-tuned on 3 sample questions

**Default model: `microsoft/phi-2` (2.7B)** - fits T4 free-tier comfortably (~2 GB VRAM), no HuggingFace token required, trains 2-3x faster than Mistral-7B. Mistral remains opt-in via the `USE_PHI2` toggle in Step 1.

**Total runtime:** ~10 min on a free T4. If every cell passes, the full project will work.

**Hardware required:** `Runtime -> Change runtime type -> T4 GPU`

---
## Smoke-test settings (reduced for speed)

| Param | Smoke-test | Production |
|---|---|---|
| Base model | Phi-2 (2.7B) | Phi-2 or Mistral-7B-Instruct |
| Training samples | 500 | 10,000 |
| Training steps | 50 | full epoch (~625) |
| Corpus size | 1,000 docs | 5,000 docs |
| Expected runtime (T4) | ~10 min | ~75 min |


---
## Step 0 — GPU sanity check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

import torch
print(f'\nCUDA available : {torch.cuda.is_available()}')
print(f'Device         : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'VRAM total     : {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB' if torch.cuda.is_available() else '')

assert torch.cuda.is_available(), 'No GPU detected — switch runtime to T4 GPU before continuing'

---
## Step 1 — Install dependencies (single atomic transaction)

Two-cell pattern:
1. **Install everything in ONE pip call** so pip's resolver picks compatible
   binaries up-front (avoids the dreaded successive-install version drift).
2. **Restart the kernel via the Jupyter API** so freshly-installed NumPy 1.x
   replaces the in-memory NumPy 2.x that ships with Colab.

After the restart, Colab will reconnect automatically (~10 sec).
**Skip both cells in Step 1 on subsequent runs — packages persist on disk.**

In [ ]:
%%capture
# ONE atomic install — every constraint resolved together. This is the
# version lock-set that worked end-to-end on Colab T4 with the AmazonQA-RAG
# project. Conflict warnings about gradio/diffusers/cupy/jax are harmless
# (those are Colab's pre-installed extras we don't use).
!pip install -q \
    "numpy==1.26.4" \
    "torch==2.4.0" \
    "transformers==4.44.2" \
    "tokenizers==0.19.1" \
    "huggingface_hub==0.25.2" \
    "peft==0.13.2" \
    "trl==0.11.4" \
    "bitsandbytes==0.43.3" \
    "accelerate==0.34.2" \
    "datasets==2.21.0" \
    "sentence-transformers==3.0.1" \
    "faiss-cpu==1.8.0.post1" \
    "langchain==0.2.16" \
    "langchain-community==0.2.16" \
    "langchain-huggingface==0.0.3" \
    "fsspec==2024.6.1" \
    "sentencepiece==0.2.0"

### Restart the kernel so the freshly-installed NumPy 1.x is loaded

Colab will show *"Kernel is restarting"* / *"Your session crashed"* — **this is intentional and expected**.
After the auto-reconnect, **skip the install cell AND this restart cell** and start running cells from the GPU check (Step 0) onward.

If the auto-restart doesn't trigger, manually do **Runtime → Restart session** in the Colab top menu — same result.

In [ ]:
# Reliable kernel restart — the official Jupyter API.
# Same effect as 'Runtime → Restart session' from the Colab menu.
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

### Verify the version lock landed

Run this cell after the kernel auto-reconnects. You want to see:
```
numpy           : 1.26.4
torch           : 2.4.0
transformers    : 4.44.2
huggingface_hub : 0.25.2
peft            : 0.13.2
trl             : 0.11.4
```
If any version is different, do **Runtime → Disconnect and delete runtime** and start over from Step 0.

In [ ]:
import numpy, torch, transformers, huggingface_hub, peft, trl
print(f'numpy           : {numpy.__version__}')
print(f'torch           : {torch.__version__}')
print(f'transformers    : {transformers.__version__}')
print(f'huggingface_hub : {huggingface_hub.__version__}')
print(f'peft            : {peft.__version__}')
print(f'trl             : {trl.__version__}')

assert numpy.__version__.startswith('1.'), 'NumPy 1.x not loaded — restart kernel via menu and retry'
print('\nAll versions match the lock-set. Continue to HuggingFace login below.')

### Authenticate with HuggingFace (only needed if you want Mistral-7B)

**Default model is Phi-2** (Microsoft, 2.7B, not gated, fits T4 free-tier easily). If you want to try Mistral-7B instead:

1. Accept the license at https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2
2. Create a read-token at https://huggingface.co/settings/tokens
3. In Colab, click the **key icon** in the left sidebar → **+ Add new secret** → name `HF_TOKEN`, value = your token, toggle **Notebook access** ON
4. Set `USE_PHI2 = False` in the cell below

**Recommendation: stick with Phi-2 for the smoke test** — Mistral-7B in 4-bit is on the edge of T4 VRAM and frequently OOMs on free-tier.

In [ ]:
# DEFAULT: Phi-2. Phi-2 is the safe choice on free-tier Colab.
# Set to False ONLY if you have a paid Colab tier OR have confirmed Mistral fits.
USE_PHI2 = True

if not USE_PHI2:
    from huggingface_hub import login
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
        login(token=hf_token)
        print('Logged in to HuggingFace via Colab secret.')
    except Exception as e:
        print(f'No HF_TOKEN secret found ({e}).')
        print('Falling back to Phi-2 — no authentication required.')
        USE_PHI2 = True

print(f'\nModel selection: {"microsoft/phi-2 (default, recommended)" if USE_PHI2 else "mistralai/Mistral-7B-Instruct-v0.2"}')

---
## Step 2 — Inline `configs/config.py`

All hyperparameters live here as dataclasses. No magic numbers anywhere else.
You can edit any value in this cell and re-run downstream cells to experiment.

In [ ]:
"""
Central configuration for the AmazonQA-RAG project.

All hyperparameters, paths, and model identifiers live here so that experiments
are reproducible and no magic numbers appear in the pipeline code.

Switching between Mistral-7B and the Phi-2 fallback is a one-line change:
    ModelConfig.base_model_id = "microsoft/phi-2"
    ModelConfig.lora_target_modules = PHI2_LORA_TARGETS
"""

from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Optional


# ---------------------------------------------------------------------------
# Path configuration
# ---------------------------------------------------------------------------
# Resolve project root from this file's location so paths work on both
# Windows (local dev) and Linux (Colab) without modification.
#
# `__file__` is undefined when this code is executed inside a Jupyter/Colab
# notebook cell (notebook cells are strings, not files), so we fall back to
# the current working directory in that case.
try:
    PROJECT_ROOT = Path(__file__).resolve().parent.parent
except NameError:
    PROJECT_ROOT = Path.cwd()

# Colab override: if /content exists we are on Colab, prefer that workspace
# so that Drive mounting / checkpoint persistence works out of the box.
if Path("/content").exists():
    PROJECT_ROOT = Path("/content/AmazonQA-RAG")
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)


@dataclass
class PathConfig:
    """File-system paths used across all phases."""

    project_root: Path = PROJECT_ROOT
    data_dir: Path = PROJECT_ROOT / "data"
    raw_data_dir: Path = PROJECT_ROOT / "data" / "raw"
    processed_data_dir: Path = PROJECT_ROOT / "data" / "processed"
    eval_data_dir: Path = PROJECT_ROOT / "data" / "eval"

    # Model artifacts
    adapter_dir: Path = PROJECT_ROOT / "artifacts" / "lora_adapter"
    merged_model_dir: Path = PROJECT_ROOT / "artifacts" / "merged_model"
    vector_store_dir: Path = PROJECT_ROOT / "artifacts" / "faiss_index"

    # Results
    results_dir: Path = PROJECT_ROOT / "results"
    logs_dir: Path = PROJECT_ROOT / "logs"

    def ensure_all(self) -> None:
        """Create every directory referenced above if it does not already exist."""
        for attr_name in self.__dataclass_fields__:
            path = getattr(self, attr_name)
            if isinstance(path, Path):
                path.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------------
# Dataset configuration
# ---------------------------------------------------------------------------
@dataclass
class DatasetConfig:
    """
    Dataset selection and preprocessing parameters.

    We default to `embedding-data/Amazon-QA` — a clean, well-formed
    Amazon question/answer corpus (~190k QA pairs). The legacy
    `amazon_qa` script-based dataset is deprecated; this one is direct-load.
    """

    hf_dataset_id: str = "embedding-data/Amazon-QA"
    hf_dataset_split: str = "train"

    # Subsample for free-tier training. Full 190k * 3 epochs would take ~12h on T4.
    # 10k samples * 1 epoch ≈ 45–60 min on T4 with seq_len=512, batch=4, grad_accum=4.
    train_sample_size: int = 10_000
    eval_sample_size: int = 500

    # Train/validation split ratio applied AFTER sub-sampling.
    val_split_ratio: float = 0.05

    # Tokenization
    max_seq_length: int = 512

    # Reproducibility
    seed: int = 42


# ---------------------------------------------------------------------------
# Model configuration
# ---------------------------------------------------------------------------
# Recommended LoRA target modules per architecture family.
# Determined by inspecting `model.named_modules()` for each base model.
MISTRAL_LORA_TARGETS: List[str] = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]
PHI2_LORA_TARGETS: List[str] = ["q_proj", "k_proj", "v_proj", "dense"]
LLAMA_LORA_TARGETS: List[str] = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]


@dataclass
class ModelConfig:
    """
    Base model + tokenizer settings.

    PRIMARY: Mistral-7B-Instruct-v0.2 (4-bit, ~5GB VRAM for weights).
    FALLBACK: microsoft/phi-2 (2.7B, ~2GB VRAM in 4-bit) — use if OOM on T4.
    """

    base_model_id: str = "mistralai/Mistral-7B-Instruct-v0.2"
    tokenizer_id: Optional[str] = None  # defaults to base_model_id when None

    # 4-bit quantization (BitsAndBytes NF4 — the QLoRA paper recipe)
    load_in_4bit: bool = True
    bnb_4bit_quant_type: str = "nf4"
    bnb_4bit_compute_dtype: str = "bfloat16"  # T4 supports bf16 via autocast
    bnb_4bit_use_double_quant: bool = True

    # Device / memory
    device_map: str = "auto"
    trust_remote_code: bool = True


@dataclass
class LoRAConfig:
    """
    Low-Rank Adaptation hyperparameters.

    rank=16, alpha=32 is the QLoRA paper default and is a sweet spot for
    7B-class models. Larger rank → more capacity but more VRAM + slower.
    """

    r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    bias: str = "none"
    task_type: str = "CAUSAL_LM"
    target_modules: List[str] = field(
        default_factory=lambda: MISTRAL_LORA_TARGETS.copy()
    )


# ---------------------------------------------------------------------------
# Training configuration
# ---------------------------------------------------------------------------
@dataclass
class TrainingConfig:
    """
    SFTTrainer / TrainingArguments hyperparameters.

    Effective batch size = per_device_train_batch_size * gradient_accumulation_steps
                         = 4 * 4 = 16
    On T4 with Mistral-7B-4bit + LoRA + seq_len=512 this stays under 14GB VRAM.
    """

    # Optimizer
    learning_rate: float = 2e-4
    weight_decay: float = 0.001
    optim: str = "paged_adamw_8bit"   # memory-efficient AdamW for QLoRA
    lr_scheduler_type: str = "cosine"
    warmup_ratio: float = 0.03
    max_grad_norm: float = 0.3

    # Batching
    per_device_train_batch_size: int = 4
    per_device_eval_batch_size: int = 4
    gradient_accumulation_steps: int = 4
    gradient_checkpointing: bool = True

    # Duration
    num_train_epochs: int = 1
    max_steps: int = -1               # -1 → train for full num_train_epochs

    # Precision (T4 supports fp16 natively; bf16 is emulated)
    fp16: bool = True
    bf16: bool = False

    # Logging / saving
    logging_steps: int = 25
    save_steps: int = 250
    save_total_limit: int = 2
    eval_strategy: str = "steps"
    eval_steps: int = 250
    report_to: str = "none"           # set to "wandb" if you want experiment tracking

    # Misc
    seed: int = 42
    group_by_length: bool = True      # speeds up training by ~20% via length bucketing


# ---------------------------------------------------------------------------
# Prompt template
# ---------------------------------------------------------------------------
# Mistral-Instruct uses the [INST] ... [/INST] format. Keeping this in config
# means we never hardcode it in trainer or inference code.
PROMPT_TEMPLATE_MISTRAL: str = (
    "<s>[INST] You are a helpful Amazon shopping assistant. "
    "Answer the customer's question accurately and concisely.\n\n"
    "Question: {question} [/INST] {answer}</s>"
)

PROMPT_TEMPLATE_PHI2: str = (
    "Instruct: You are a helpful Amazon shopping assistant. "
    "Answer the customer's question accurately and concisely.\n"
    "Question: {question}\n"
    "Output: {answer}"
)


# ---------------------------------------------------------------------------
# RAG prompt templates (Phase 2)
# ---------------------------------------------------------------------------
# The RAG prompt MUST do three things well:
#   1. Clearly delimit retrieved context from the question (we use ### markers).
#   2. Instruct the model to ground its answer in the context.
#   3. Tell it what to do when the context is insufficient (avoid hallucination).
# These three lines together are the single biggest lever on RAGAS faithfulness.
RAG_PROMPT_TEMPLATE_MISTRAL: str = (
    "<s>[INST] You are a helpful Amazon shopping assistant. Use ONLY the context "
    "below to answer the customer's question. If the context does not contain "
    "the answer, say \"I don't have enough information to answer that.\" — do "
    "not invent product details, prices, or compatibility claims.\n\n"
    "### Context\n{context}\n\n"
    "### Question\n{question} [/INST]"
)

RAG_PROMPT_TEMPLATE_PHI2: str = (
    "Instruct: You are a helpful Amazon shopping assistant. Use ONLY the context "
    "below to answer the customer's question. If the context does not contain "
    "the answer, say \"I don't have enough information to answer that.\"\n\n"
    "### Context\n{context}\n\n"
    "### Question\n{question}\n"
    "Output:"
)


# ---------------------------------------------------------------------------
# RAG / retrieval configuration
# ---------------------------------------------------------------------------
@dataclass
class RAGConfig:
    """
    Retrieval Augmented Generation hyperparameters.

    Defaults are tuned for Amazon-style short QA: chunk_size=512 captures
    a typical product Q+A turn without wasting context window; top_k=5
    gives the LLM enough evidence to triangulate without blowing the prompt.
    """

    # -- Embedding model --
    # all-MiniLM-L6-v2: 384-dim, ~80 MB, 14k sentences/sec on CPU.
    # Strong recall on short QA, far cheaper than e5-large or bge-large.
    embedding_model_id: str = "sentence-transformers/all-MiniLM-L6-v2"
    embedding_device: str = "cuda"        # auto-falls-back to cpu in code
    embedding_normalize: bool = True      # cosine similarity via inner-product

    # -- Document loading --
    # Where the source corpus lives. Either a CSV with `question,answer` cols,
    # a JSON-Lines file with the same fields, or a directory of .txt files.
    corpus_path: Optional[str] = None     # set by load helper if None
    corpus_format: str = "auto"           # "csv" | "jsonl" | "txt" | "auto"
    corpus_text_columns: List[str] = field(
        default_factory=lambda: ["question", "answer"]
    )
    # Subsample for fast indexing during development. None = use all rows.
    corpus_sample_size: Optional[int] = 5000

    # -- Chunking --
    chunk_size: int = 512                 # characters, not tokens
    chunk_overlap: int = 50
    chunk_separators: List[str] = field(
        default_factory=lambda: ["\n\n", "\n", ". ", " ", ""]
    )

    # -- Retrieval --
    top_k: int = 5
    search_type: str = "similarity"       # "similarity" | "mmr"
    # MMR settings (only used if search_type == "mmr") — encourages diverse
    # results so the LLM isn't shown 5 near-duplicate chunks.
    mmr_fetch_k: int = 20
    mmr_lambda: float = 0.5

    # -- Generation --
    # Lower temperature for RAG: we want the model to stick to the context.
    max_new_tokens: int = 256
    temperature: float = 0.3
    top_p: float = 0.9
    repetition_penalty: float = 1.1

    # -- Vector store --
    # FAISS index type. "Flat" is exact + small-corpus optimal.
    # Switch to "IVFFlat" with nlist=100 for corpora > 100k chunks.
    faiss_index_type: str = "Flat"

    # -- Prompt --
    rag_prompt_template: str = RAG_PROMPT_TEMPLATE_MISTRAL


# ---------------------------------------------------------------------------
# Convenience aggregator
# ---------------------------------------------------------------------------
@dataclass
class FineTuneConfig:
    """One-stop bundle passed around between modules."""

    paths: PathConfig = field(default_factory=PathConfig)
    dataset: DatasetConfig = field(default_factory=DatasetConfig)
    model: ModelConfig = field(default_factory=ModelConfig)
    lora: LoRAConfig = field(default_factory=LoRAConfig)
    training: TrainingConfig = field(default_factory=TrainingConfig)
    rag: RAGConfig = field(default_factory=RAGConfig)
    prompt_template: str = PROMPT_TEMPLATE_MISTRAL


def get_default_config() -> FineTuneConfig:
    """Return the default project configuration with directories pre-created."""
    cfg = FineTuneConfig()
    cfg.paths.ensure_all()
    return cfg


def get_phi2_fallback_config() -> FineTuneConfig:
    """
    Return a configuration that swaps in Phi-2 (2.7B) for the base model.

    Use this if Mistral-7B OOMs on your T4 instance — Phi-2 in 4-bit needs
    only ~2GB for weights and trains 2-3x faster.
    """
    cfg = get_default_config()
    cfg.model.base_model_id = "microsoft/phi-2"
    cfg.lora.target_modules = PHI2_LORA_TARGETS.copy()
    cfg.prompt_template = PROMPT_TEMPLATE_PHI2
    cfg.rag.rag_prompt_template = RAG_PROMPT_TEMPLATE_PHI2
    cfg.training.per_device_train_batch_size = 8  # Phi-2 is smaller → bigger batches
    cfg.training.gradient_accumulation_steps = 2
    return cfg

### Load the right config based on the Phi-2 toggle from Step 1

In [ ]:
cfg = get_phi2_fallback_config() if USE_PHI2 else get_default_config()

# Smoke-test overrides — DRASTICALLY reduce training time for verification.
# Remove or raise these values for a real run.
cfg.dataset.train_sample_size = 500
cfg.dataset.val_split_ratio = 0.1
cfg.training.max_steps = 50          # ~5 min instead of ~60
cfg.training.logging_steps = 5
cfg.training.save_steps = 50
cfg.training.eval_steps = 50
cfg.rag.corpus_sample_size = 1000

print(f'Base model       : {cfg.model.base_model_id}')
print(f'Train samples    : {cfg.dataset.train_sample_size}')
print(f'Max train steps  : {cfg.training.max_steps}')
print(f'Corpus size      : {cfg.rag.corpus_sample_size}')
print(f'Adapter out dir  : {cfg.paths.adapter_dir}')

---
## Step 3 — Inline `src/finetune.py` (Phase 1 engine)

Defines: dataset loading + preprocessing, 4-bit base model loading,
LoRA attachment, SFTTrainer construction, and inference helpers.

In [ ]:
"""
QLoRA fine-tuning engine for the AmazonQA-RAG project.

This module wraps the full Phase 1 pipeline:
  1. Load + preprocess the Amazon QA dataset
  2. Load the base LLM in 4-bit (BitsAndBytes NF4)
  3. Attach LoRA adapters (PEFT)
  4. Train with SFTTrainer (TRL)
  5. Save adapter weights + run a smoke-test generation

It is imported by `notebooks/01_qlora_finetune.ipynb` and can also be invoked
directly from the CLI:  python -m src.finetune

The notebook contains the narrative; this file contains the logic.
"""

from __future__ import annotations

import gc
import logging
from pathlib import Path
from typing import Dict, Optional, Tuple


import torch
from datasets import Dataset, DatasetDict, load_dataset
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from trl import SFTTrainer


# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------
logging.basicConfig(
    format="[%(asctime)s] %(levelname)s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
    level=logging.INFO,
)
logger = logging.getLogger("amazonqa.finetune")


# ===========================================================================
# 1. Data loading + preprocessing
# ===========================================================================
def load_and_preprocess_dataset(cfg: FineTuneConfig) -> DatasetDict:
    """
    Load the Amazon QA dataset from HuggingFace and format it for SFT.

    The `embedding-data/Amazon-QA` dataset ships rows of the form:
        {"query": "...", "pos": ["answer 1", "answer 2", ...]}
    We flatten this to (question, answer) pairs and then apply the
    instruction-tuning prompt template defined in the config.

    Args:
        cfg: Project configuration.

    Returns:
        A `DatasetDict` with `train` and `validation` splits, each row
        containing a single `text` column ready for SFTTrainer.

    Raises:
        RuntimeError: If the dataset cannot be downloaded.
    """
    logger.info("Downloading dataset: %s", cfg.dataset.hf_dataset_id)
    try:
        raw = load_dataset(cfg.dataset.hf_dataset_id, split=cfg.dataset.hf_dataset_split)
    except Exception as exc:
        raise RuntimeError(
            f"Failed to load dataset '{cfg.dataset.hf_dataset_id}'. "
            f"Check your internet connection and HuggingFace cache. Original error: {exc}"
        ) from exc

    logger.info("Raw dataset loaded: %d rows", len(raw))

    # ------------------------------------------------------------------
    # Sub-sample BEFORE flattening to keep memory low
    # ------------------------------------------------------------------
    sample_size = min(cfg.dataset.train_sample_size, len(raw))
    raw = raw.shuffle(seed=cfg.dataset.seed).select(range(sample_size))

    # ------------------------------------------------------------------
    # Flatten {query, pos:[...]} → list of {question, answer}
    # ------------------------------------------------------------------
    def _flatten(example: Dict) -> Dict:
        question = example.get("query") or example.get("question") or ""
        answers = example.get("pos") or example.get("answers") or []
        # Use the first (highest-quality) answer
        answer = answers[0] if isinstance(answers, list) and answers else ""
        return {"question": question.strip(), "answer": answer.strip()}

    flat = raw.map(_flatten, remove_columns=raw.column_names)
    flat = flat.filter(lambda r: len(r["question"]) > 5 and len(r["answer"]) > 5)

    logger.info("After flatten + filter: %d rows", len(flat))

    # ------------------------------------------------------------------
    # Apply the instruction prompt template
    # ------------------------------------------------------------------
    def _format(example: Dict) -> Dict:
        text = cfg.prompt_template.format(
            question=example["question"], answer=example["answer"]
        )
        return {"text": text}

    formatted = flat.map(_format, remove_columns=flat.column_names)

    # ------------------------------------------------------------------
    # Train / validation split
    # ------------------------------------------------------------------
    splits = formatted.train_test_split(
        test_size=cfg.dataset.val_split_ratio, seed=cfg.dataset.seed
    )
    ds = DatasetDict(train=splits["train"], validation=splits["test"])
    logger.info(
        "Final split → train: %d | validation: %d",
        len(ds["train"]),
        len(ds["validation"]),
    )

    # Persist a copy for reproducibility
    ds.save_to_disk(str(cfg.paths.processed_data_dir / "amazon_qa_sft"))
    return ds


# ===========================================================================
# 2. Model + tokenizer loading
# ===========================================================================
def load_base_model_and_tokenizer(
    cfg: FineTuneConfig,
) -> Tuple[AutoModelForCausalLM, AutoTokenizer]:
    """
    Load the base LLM in 4-bit and its tokenizer.

    Uses the QLoRA recipe: NF4 quantization + double-quant + bf16 compute dtype.
    Gradient checkpointing is enabled by the trainer, so we call
    `prepare_model_for_kbit_training` to wire up the necessary hooks.

    Returns:
        Tuple of (4-bit-quantized model ready for LoRA, tokenizer).
    """
    logger.info("Configuring BitsAndBytes 4-bit quantization (NF4)")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=cfg.model.load_in_4bit,
        bnb_4bit_quant_type=cfg.model.bnb_4bit_quant_type,
        bnb_4bit_compute_dtype=getattr(torch, cfg.model.bnb_4bit_compute_dtype),
        bnb_4bit_use_double_quant=cfg.model.bnb_4bit_use_double_quant,
    )

    logger.info("Loading base model: %s", cfg.model.base_model_id)
    try:
        model = AutoModelForCausalLM.from_pretrained(
            cfg.model.base_model_id,
            quantization_config=bnb_config,
            device_map=cfg.model.device_map,
            trust_remote_code=cfg.model.trust_remote_code,
        )
    except torch.cuda.OutOfMemoryError as exc:
        raise RuntimeError(
            "OOM while loading the base model. Options:\n"
            "  1. Use the Phi-2 fallback config (get_phi2_fallback_config())\n"
            "  2. Restart the Colab runtime to free leaked memory\n"
            "  3. Reduce max_seq_length in DatasetConfig"
        ) from exc

    model.config.use_cache = False        # required for gradient checkpointing
    model.config.pretraining_tp = 1       # disable tensor parallelism slicing

    logger.info("Preparing model for k-bit training")
    model = prepare_model_for_kbit_training(model)

    # ------------------------------------------------------------------
    # Tokenizer
    # ------------------------------------------------------------------
    tokenizer_id = cfg.model.tokenizer_id or cfg.model.base_model_id
    logger.info("Loading tokenizer: %s", tokenizer_id)
    tokenizer = AutoTokenizer.from_pretrained(
        tokenizer_id, trust_remote_code=cfg.model.trust_remote_code
    )
    # Causal LMs train left→right; padding must go on the right during SFT
    # so loss isn't computed on padding tokens that bleed into the answer.
    tokenizer.padding_side = "right"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    return model, tokenizer


# ===========================================================================
# 3. LoRA attachment
# ===========================================================================
def attach_lora_adapters(
    model: AutoModelForCausalLM, cfg: FineTuneConfig
) -> AutoModelForCausalLM:
    """
    Wrap the base model with LoRA adapters using PEFT.

    Only the adapter weights become trainable; the 4-bit base stays frozen,
    which is what makes QLoRA fit on consumer GPUs.

    Returns:
        PEFT-wrapped model with trainable LoRA parameters.
    """
    logger.info(
        "Attaching LoRA: r=%d, alpha=%d, dropout=%.2f, targets=%s",
        cfg.lora.r,
        cfg.lora.lora_alpha,
        cfg.lora.lora_dropout,
        cfg.lora.target_modules,
    )
    lora_config = LoraConfig(
        r=cfg.lora.r,
        lora_alpha=cfg.lora.lora_alpha,
        lora_dropout=cfg.lora.lora_dropout,
        bias=cfg.lora.bias,
        task_type=cfg.lora.task_type,
        target_modules=cfg.lora.target_modules,
    )
    model = get_peft_model(model, lora_config)

    # Report trainable param count — should be ~0.1–1% of total for QLoRA
    trainable, total = 0, 0
    for _, p in model.named_parameters():
        total += p.numel()
        if p.requires_grad:
            trainable += p.numel()
    logger.info(
        "Trainable params: %s / %s  (%.4f%%)",
        f"{trainable:,}",
        f"{total:,}",
        100 * trainable / total,
    )
    return model


# ===========================================================================
# 4. Training
# ===========================================================================
def build_trainer(
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    dataset: DatasetDict,
    cfg: FineTuneConfig,
) -> SFTTrainer:
    """Construct a TRL SFTTrainer wired with our hyperparameters."""
    logger.info("Building TrainingArguments")
    training_args = TrainingArguments(
        output_dir=str(cfg.paths.adapter_dir),
        num_train_epochs=cfg.training.num_train_epochs,
        max_steps=cfg.training.max_steps,
        per_device_train_batch_size=cfg.training.per_device_train_batch_size,
        per_device_eval_batch_size=cfg.training.per_device_eval_batch_size,
        gradient_accumulation_steps=cfg.training.gradient_accumulation_steps,
        gradient_checkpointing=cfg.training.gradient_checkpointing,
        optim=cfg.training.optim,
        learning_rate=cfg.training.learning_rate,
        weight_decay=cfg.training.weight_decay,
        lr_scheduler_type=cfg.training.lr_scheduler_type,
        warmup_ratio=cfg.training.warmup_ratio,
        max_grad_norm=cfg.training.max_grad_norm,
        fp16=cfg.training.fp16,
        bf16=cfg.training.bf16,
        logging_steps=cfg.training.logging_steps,
        save_steps=cfg.training.save_steps,
        save_total_limit=cfg.training.save_total_limit,
        eval_strategy=cfg.training.eval_strategy,
        eval_steps=cfg.training.eval_steps,
        report_to=cfg.training.report_to,
        seed=cfg.training.seed,
        group_by_length=cfg.training.group_by_length,
        logging_dir=str(cfg.paths.logs_dir),
    )

    logger.info("Building SFTTrainer")
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"],
        dataset_text_field="text",
        max_seq_length=cfg.dataset.max_seq_length,
        args=training_args,
        packing=False,  # packing speeds up training but hurts loss on QA-style data
    )
    return trainer


def train(cfg: Optional[FineTuneConfig] = None) -> Path:
    """
    Run the full Phase 1 pipeline end-to-end.

    Args:
        cfg: Optional override config. Defaults to `get_default_config()`.

    Returns:
        Path to the saved LoRA adapter directory.
    """
    cfg = cfg or get_default_config()
    cfg.paths.ensure_all()

    # 1. Data
    dataset = load_and_preprocess_dataset(cfg)

    # 2. Model + tokenizer
    model, tokenizer = load_base_model_and_tokenizer(cfg)

    # 3. LoRA
    model = attach_lora_adapters(model, cfg)

    # 4. Trainer
    trainer = build_trainer(model, tokenizer, dataset, cfg)

    logger.info("Starting training")
    trainer.train()

    # 5. Save adapter (NOT the full model — only the small LoRA weights)
    logger.info("Saving LoRA adapter to %s", cfg.paths.adapter_dir)
    trainer.model.save_pretrained(str(cfg.paths.adapter_dir))
    tokenizer.save_pretrained(str(cfg.paths.adapter_dir))

    # 6. Free memory before returning
    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()

    return cfg.paths.adapter_dir


# ===========================================================================
# 5. Smoke-test inference (used by the notebook for before/after comparison)
# ===========================================================================
def generate_with_model(
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    prompt: str,
    max_new_tokens: int = 200,
    temperature: float = 0.7,
    top_p: float = 0.9,
) -> str:
    """
    Run a single forward generation pass and return the decoded completion.

    Used both for the "before fine-tuning" baseline and "after fine-tuning"
    comparison in the notebook.
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Strip the echoed prompt
    if text.startswith(prompt):
        text = text[len(prompt):].strip()
    return text


def load_finetuned_model(
    cfg: FineTuneConfig,
) -> Tuple[PeftModel, AutoTokenizer]:
    """
    Reload the base model + apply the saved LoRA adapter for inference.

    This is the canonical way to use the fine-tuned model in downstream
    phases (RAG, evaluation, Streamlit app).
    """
    base, tokenizer = load_base_model_and_tokenizer(cfg)
    logger.info("Loading LoRA adapter from %s", cfg.paths.adapter_dir)
    model = PeftModel.from_pretrained(base, str(cfg.paths.adapter_dir))
    model.eval()
    return model, tokenizer


# ===========================================================================
# CLI entry point
# ===========================================================================
if __name__ == "__main__":
    adapter_path = train()
    print(f"\nDone. Adapter saved to: {adapter_path}")

---
## Step 4 — Run Phase 1: load data + base model + LoRA + train

This is the longest cell — ~5 min with smoke-test settings, ~60 min for full run.

In [ ]:
# 4.1 Load and preprocess dataset
dataset = load_and_preprocess_dataset(cfg)
print(dataset)
print('\nFormatted example:\n', dataset['train'][0]['text'][:400])

In [ ]:
# 4.2 Load 4-bit base model + tokenizer
model, tokenizer = load_base_model_and_tokenizer(cfg)
import torch
print(f'\nVRAM after base load: {torch.cuda.memory_allocated()/1024**3:.2f} GB')

In [ ]:
# 4.3 Baseline generation BEFORE fine-tuning — keep for comparison
SAMPLE_Q = 'Does the Echo Dot 5th gen work with Apple Music?'
baseline_prompt = cfg.prompt_template.format(question=SAMPLE_Q, answer='').rstrip()
baseline_answer = generate_with_model(model, tokenizer, baseline_prompt, max_new_tokens=120)
print('=== BEFORE fine-tuning ===\n', baseline_answer)

In [ ]:
# 4.4 Attach LoRA adapters
model = attach_lora_adapters(model, cfg)
model.print_trainable_parameters()

In [ ]:
# 4.5 Build trainer
trainer = build_trainer(model, tokenizer, dataset, cfg)

In [ ]:
# 4.6 Train (this is the long step — ~5 min smoke test, ~60 min full)
trainer.train()

In [ ]:
# 4.7 Save adapter to disk
trainer.model.save_pretrained(str(cfg.paths.adapter_dir))
tokenizer.save_pretrained(str(cfg.paths.adapter_dir))
!ls -lh {cfg.paths.adapter_dir}

In [ ]:
# 4.8 Generate AFTER fine-tuning with the SAME prompt → side-by-side
ft_answer = generate_with_model(trainer.model, tokenizer, baseline_prompt, max_new_tokens=120)
print('=== BEFORE ===\n', baseline_answer)
print('\n=== AFTER  ===\n', ft_answer)

In [ ]:
# 4.9 Free GPU before Phase 2
import gc, torch
del trainer, model
gc.collect()
torch.cuda.empty_cache()
print(f'VRAM after cleanup: {torch.cuda.memory_allocated()/1024**3:.2f} GB')

---
## Step 5 — Inline `src/llm_loader.py`

Wraps the base + adapter into a LangChain-compatible `HuggingFacePipeline`.
Used by both the RAG pipeline (Step 7) and the Streamlit app (Phase 4).

In [ ]:
"""
LLM loader — produces LangChain-compatible LLM wrappers around our models.

Three loading modes are supported, all behind one stable interface:

    load_llm(cfg, mode="base")        → base instruction-tuned model only
    load_llm(cfg, mode="finetuned")   → base + our QLoRA adapter
    load_llm(cfg, mode="auto")        → finetuned if adapter exists, else base

This single module is the seam between Phase 1 (raw PyTorch + PEFT) and
Phases 2-4 (LangChain pipelines + Streamlit). Keeping it isolated means
the rest of the project never imports peft/transformers directly.
"""

from __future__ import annotations

import logging
from pathlib import Path
from typing import Literal, Optional, Tuple


import torch
from peft import PeftModel
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    pipeline,
)

# LangChain has reshuffled this import path twice in the last year.
# We try the newest location first, then fall back gracefully.
try:
    from langchain_huggingface import HuggingFacePipeline
except ImportError:  # pragma: no cover - older langchain
    try:
        from langchain_community.llms import HuggingFacePipeline  # type: ignore
    except ImportError:
        from langchain.llms import HuggingFacePipeline  # type: ignore


logger = logging.getLogger("amazonqa.llm_loader")


# ---------------------------------------------------------------------------
# Internal helpers
# ---------------------------------------------------------------------------
def _build_bnb_config(cfg: FineTuneConfig) -> Optional[BitsAndBytesConfig]:
    """Return a BitsAndBytes 4-bit config, or None if CUDA isn't available."""
    if not torch.cuda.is_available():
        logger.warning(
            "CUDA unavailable — loading in full precision on CPU. This will "
            "be slow and is only intended for code-path testing."
        )
        return None

    return BitsAndBytesConfig(
        load_in_4bit=cfg.model.load_in_4bit,
        bnb_4bit_quant_type=cfg.model.bnb_4bit_quant_type,
        bnb_4bit_compute_dtype=getattr(torch, cfg.model.bnb_4bit_compute_dtype),
        bnb_4bit_use_double_quant=cfg.model.bnb_4bit_use_double_quant,
    )


def _load_tokenizer(cfg: FineTuneConfig) -> AutoTokenizer:
    """Load and configure the tokenizer for inference (left-padding for generate)."""
    tokenizer_id = cfg.model.tokenizer_id or cfg.model.base_model_id
    tokenizer = AutoTokenizer.from_pretrained(
        tokenizer_id, trust_remote_code=cfg.model.trust_remote_code
    )
    # Generation uses left-padding so newly generated tokens append correctly.
    # (Training used right-padding — different requirement.)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer


def _adapter_exists(cfg: FineTuneConfig) -> bool:
    """Check whether the LoRA adapter from Phase 1 is on disk."""
    adapter_cfg = cfg.paths.adapter_dir / "adapter_config.json"
    return adapter_cfg.exists()


# ---------------------------------------------------------------------------
# Public API — raw PyTorch model + tokenizer
# ---------------------------------------------------------------------------
def load_raw_model(
    cfg: FineTuneConfig,
    mode: Literal["base", "finetuned", "auto"] = "auto",
) -> Tuple[torch.nn.Module, AutoTokenizer]:
    """
    Load the base LLM (and optionally apply the QLoRA adapter).

    This returns raw transformers objects — use `load_llm()` for the
    LangChain-wrapped version.

    Args:
        cfg: Project config.
        mode: 'base' → just the base model; 'finetuned' → base + adapter
              (raises if adapter missing); 'auto' → finetuned if available
              else base with a warning.

    Returns:
        (model, tokenizer) ready for inference.
    """
    # Resolve auto-mode
    if mode == "auto":
        mode = "finetuned" if _adapter_exists(cfg) else "base"
        logger.info("Auto-selected mode: %s", mode)

    bnb_config = _build_bnb_config(cfg)
    logger.info("Loading base model: %s", cfg.model.base_model_id)

    try:
        model = AutoModelForCausalLM.from_pretrained(
            cfg.model.base_model_id,
            quantization_config=bnb_config,
            device_map=cfg.model.device_map if torch.cuda.is_available() else "cpu",
            trust_remote_code=cfg.model.trust_remote_code,
        )
    except torch.cuda.OutOfMemoryError as exc:
        raise RuntimeError(
            "OOM while loading the base model. Try the Phi-2 fallback config "
            "(get_phi2_fallback_config()) or restart your Colab runtime."
        ) from exc

    model.config.use_cache = True       # re-enable KV cache for fast inference

    # ------------------------------------------------------------------
    # Apply LoRA adapter if requested
    # ------------------------------------------------------------------
    if mode == "finetuned":
        if not _adapter_exists(cfg):
            raise FileNotFoundError(
                f"No LoRA adapter found at {cfg.paths.adapter_dir}. "
                "Run Phase 1 (notebooks/01_qlora_finetune.ipynb) first, "
                "or pass mode='base' to use the unfine-tuned model."
            )
        logger.info("Applying LoRA adapter from %s", cfg.paths.adapter_dir)
        model = PeftModel.from_pretrained(model, str(cfg.paths.adapter_dir))
        # Merging adapter into base weights would be faster at inference
        # but blocks future re-loading. We keep them separate.

    model.eval()
    tokenizer = _load_tokenizer(cfg)
    return model, tokenizer


# ---------------------------------------------------------------------------
# Public API — LangChain-wrapped LLM (this is what RAG / eval / app use)
# ---------------------------------------------------------------------------
def load_llm(
    cfg: Optional[FineTuneConfig] = None,
    mode: Literal["base", "finetuned", "auto"] = "auto",
    *,
    max_new_tokens: Optional[int] = None,
    temperature: Optional[float] = None,
) -> HuggingFacePipeline:
    """
    Return a LangChain-compatible LLM ready to plug into any chain.

    Args:
        cfg: Project config. Defaults to `get_default_config()`.
        mode: see `load_raw_model`.
        max_new_tokens: override `cfg.rag.max_new_tokens`.
        temperature: override `cfg.rag.temperature`.

    Returns:
        A `HuggingFacePipeline` LLM. Call `.invoke(prompt)` to generate.
    """
    cfg = cfg or get_default_config()

    model, tokenizer = load_raw_model(cfg, mode=mode)

    gen_pipe = pipeline(
        task="text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=max_new_tokens or cfg.rag.max_new_tokens,
        temperature=temperature if temperature is not None else cfg.rag.temperature,
        top_p=cfg.rag.top_p,
        repetition_penalty=cfg.rag.repetition_penalty,
        do_sample=True,
        return_full_text=False,    # critical — return ONLY the completion
        pad_token_id=tokenizer.pad_token_id,
    )

    llm = HuggingFacePipeline(pipeline=gen_pipe)
    logger.info(
        "LLM ready (mode=%s, max_new_tokens=%d, temperature=%.2f)",
        mode,
        gen_pipe.model.config.max_position_embeddings if hasattr(gen_pipe.model.config, 'max_position_embeddings') else -1,
        gen_pipe._forward_params.get("temperature", -1) if hasattr(gen_pipe, '_forward_params') else -1,
    )
    return llm


__all__ = ["load_raw_model", "load_llm"]

---
## Step 6 — Inline `src/rag_pipeline.py`

Document loading, chunking, FAISS indexing, retrieval, and the `RetrievalQA`
chain assembly. The `build_full_pipeline()` helper at the bottom does it all
in one call.

In [ ]:
"""
RAG pipeline for the AmazonQA-RAG project.

Builds, persists, and queries a FAISS vector index over an Amazon
product/support corpus, then wires it into a LangChain `RetrievalQA`
chain on top of the (optionally fine-tuned) LLM from Phase 1.

Public API
----------
    build_corpus_from_hf(cfg)         → quickly bootstrap a corpus from the
                                        same Amazon-QA dataset we trained on
    load_documents(cfg)               → load CSV / JSONL / TXT into LangChain Documents
    chunk_documents(docs, cfg)        → RecursiveCharacterTextSplitter
    build_vector_store(chunks, cfg)   → embeds + writes FAISS index to disk
    load_vector_store(cfg)            → reads FAISS index from disk
    build_retriever(vs, cfg)          → returns a top-k retriever
    build_rag_chain(llm, retriever, cfg) → returns a LangChain RetrievalQA
    answer(chain, question)           → returns {answer, source_documents}

The notebook `02_build_rag.ipynb` calls these in order; Phases 3 and 4
re-use the same functions.
"""

from __future__ import annotations

import json
import logging
from pathlib import Path
from typing import Any, Dict, List, Optional


import torch
from datasets import load_dataset

# LangChain imports — try modern paths first, fall back for older installs.
try:
    from langchain_community.document_loaders import (
        CSVLoader,
        JSONLoader,
        TextLoader,
        DirectoryLoader,
    )
    from langchain_community.vectorstores import FAISS
except ImportError:  # pragma: no cover
    from langchain.document_loaders import (  # type: ignore
        CSVLoader,
        JSONLoader,
        TextLoader,
        DirectoryLoader,
    )
    from langchain.vectorstores import FAISS  # type: ignore

try:
    from langchain_huggingface import HuggingFaceEmbeddings
except ImportError:  # pragma: no cover
    try:
        from langchain_community.embeddings import HuggingFaceEmbeddings  # type: ignore
    except ImportError:
        from langchain.embeddings import HuggingFaceEmbeddings  # type: ignore

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA


logger = logging.getLogger("amazonqa.rag")


# ===========================================================================
# 1. Corpus bootstrap (lets users skip "find me an Amazon dataset" friction)
# ===========================================================================
def build_corpus_from_hf(
    cfg: FineTuneConfig,
    output_format: str = "csv",
    overwrite: bool = False,
) -> Path:
    """
    Create a local Amazon QA corpus by re-using the HF dataset from Phase 1.

    This gives users an instant working corpus without hunting for files.
    For a "real" deployment you'd replace this with your own product
    catalog, help-center articles, or seller-Q&A export.

    Args:
        cfg: Project config.
        output_format: "csv" or "jsonl".
        overwrite: If False and the corpus exists, skip regeneration.

    Returns:
        Path to the written corpus file.
    """
    suffix = ".csv" if output_format == "csv" else ".jsonl"
    out_path = cfg.paths.raw_data_dir / f"amazon_corpus{suffix}"

    if out_path.exists() and not overwrite:
        logger.info("Corpus already exists at %s (use overwrite=True to regenerate)", out_path)
        return out_path

    logger.info("Bootstrapping corpus from HF dataset: %s", cfg.dataset.hf_dataset_id)
    raw = load_dataset(cfg.dataset.hf_dataset_id, split=cfg.dataset.hf_dataset_split)

    sample_size = cfg.rag.corpus_sample_size or len(raw)
    sample_size = min(sample_size, len(raw))
    raw = raw.shuffle(seed=cfg.dataset.seed).select(range(sample_size))

    # Flatten into the schema RAG expects
    rows = []
    for ex in raw:
        question = (ex.get("query") or ex.get("question") or "").strip()
        answers = ex.get("pos") or ex.get("answers") or []
        answer = (answers[0] if isinstance(answers, list) and answers else "").strip()
        if len(question) > 5 and len(answer) > 5:
            rows.append({"question": question, "answer": answer})

    cfg.paths.raw_data_dir.mkdir(parents=True, exist_ok=True)

    if output_format == "csv":
        import csv
        with out_path.open("w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=["question", "answer"])
            writer.writeheader()
            writer.writerows(rows)
    else:
        with out_path.open("w", encoding="utf-8") as f:
            for r in rows:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")

    logger.info("Wrote %d rows to %s", len(rows), out_path)
    return out_path


# ===========================================================================
# 2. Document loading (format-agnostic)
# ===========================================================================
def _detect_format(path: Path) -> str:
    """Infer corpus format from file extension."""
    ext = path.suffix.lower()
    if ext == ".csv":
        return "csv"
    if ext in (".json", ".jsonl"):
        return "jsonl"
    if ext == ".txt":
        return "txt"
    if path.is_dir():
        return "txt"  # DirectoryLoader for a folder of .txts
    raise ValueError(f"Cannot auto-detect format for {path}")


def load_documents(cfg: FineTuneConfig) -> List[Document]:
    """
    Load the source corpus into a list of LangChain `Document` objects.

    Resolution order for the corpus path:
        1. `cfg.rag.corpus_path` if set
        2. `<raw_data_dir>/amazon_corpus.csv` if it exists
        3. Auto-bootstrap from HF dataset (calls `build_corpus_from_hf`)

    Args:
        cfg: Project config.

    Returns:
        List of `Document`s with `page_content` and `metadata={"source": ...}`.
    """
    # 1. Resolve the corpus path
    if cfg.rag.corpus_path:
        corpus_path = Path(cfg.rag.corpus_path)
    elif (cfg.paths.raw_data_dir / "amazon_corpus.csv").exists():
        corpus_path = cfg.paths.raw_data_dir / "amazon_corpus.csv"
    else:
        logger.info("No corpus found — bootstrapping from HF dataset...")
        corpus_path = build_corpus_from_hf(cfg, output_format="csv")

    if not corpus_path.exists():
        raise FileNotFoundError(f"Corpus path does not exist: {corpus_path}")

    fmt = cfg.rag.corpus_format if cfg.rag.corpus_format != "auto" else _detect_format(corpus_path)
    logger.info("Loading corpus: %s (format=%s)", corpus_path, fmt)

    # 2. Load with the right loader
    if fmt == "csv":
        # CSVLoader concatenates all columns by default — we want to control
        # the joining so chunks contain both question and answer context.
        import csv as _csv
        docs: List[Document] = []
        with corpus_path.open("r", encoding="utf-8") as f:
            reader = _csv.DictReader(f)
            for i, row in enumerate(reader):
                content_parts = []
                for col in cfg.rag.corpus_text_columns:
                    if col in row and row[col]:
                        content_parts.append(f"{col.title()}: {row[col]}")
                content = "\n".join(content_parts)
                if content.strip():
                    docs.append(
                        Document(
                            page_content=content,
                            metadata={"source": str(corpus_path), "row": i},
                        )
                    )
    elif fmt == "jsonl":
        docs = []
        with corpus_path.open("r", encoding="utf-8") as f:
            for i, line in enumerate(f):
                obj = json.loads(line)
                content_parts = [
                    f"{col.title()}: {obj[col]}"
                    for col in cfg.rag.corpus_text_columns
                    if col in obj and obj[col]
                ]
                content = "\n".join(content_parts)
                if content.strip():
                    docs.append(
                        Document(
                            page_content=content,
                            metadata={"source": str(corpus_path), "row": i},
                        )
                    )
    elif fmt == "txt":
        if corpus_path.is_dir():
            loader = DirectoryLoader(
                str(corpus_path), glob="**/*.txt", loader_cls=TextLoader
            )
        else:
            loader = TextLoader(str(corpus_path), encoding="utf-8")
        docs = loader.load()
    else:
        raise ValueError(f"Unsupported corpus_format: {fmt}")

    logger.info("Loaded %d documents", len(docs))
    return docs


# ===========================================================================
# 3. Chunking
# ===========================================================================
def chunk_documents(
    docs: List[Document], cfg: FineTuneConfig
) -> List[Document]:
    """
    Split documents into overlapping chunks using RecursiveCharacterTextSplitter.

    The recursive splitter is the right default for mixed-structure text: it
    tries to split on paragraph breaks first, then sentences, then words —
    preserving semantic boundaries far better than naive fixed-length splits.

    Args:
        docs: Output of `load_documents()`.
        cfg: Project config (uses cfg.rag.chunk_size and chunk_overlap).

    Returns:
        List of chunked Documents, each ≤ chunk_size characters.
    """
    logger.info(
        "Chunking: size=%d, overlap=%d", cfg.rag.chunk_size, cfg.rag.chunk_overlap
    )
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=cfg.rag.chunk_size,
        chunk_overlap=cfg.rag.chunk_overlap,
        separators=cfg.rag.chunk_separators,
        length_function=len,
    )
    chunks = splitter.split_documents(docs)
    logger.info("Produced %d chunks from %d documents", len(chunks), len(docs))
    return chunks


# ===========================================================================
# 4. Embeddings + FAISS vector store
# ===========================================================================
def _get_embeddings(cfg: FineTuneConfig) -> HuggingFaceEmbeddings:
    """Instantiate the sentence-transformers embedding model."""
    device = cfg.rag.embedding_device
    if device == "cuda" and not torch.cuda.is_available():
        logger.warning("CUDA requested for embeddings but unavailable — using CPU.")
        device = "cpu"

    logger.info(
        "Loading embedding model: %s (device=%s)",
        cfg.rag.embedding_model_id,
        device,
    )
    return HuggingFaceEmbeddings(
        model_name=cfg.rag.embedding_model_id,
        model_kwargs={"device": device},
        encode_kwargs={"normalize_embeddings": cfg.rag.embedding_normalize},
    )


def build_vector_store(
    chunks: List[Document], cfg: FineTuneConfig, persist: bool = True
) -> FAISS:
    """
    Embed the chunks and build a FAISS index.

    Args:
        chunks: Output of `chunk_documents()`.
        cfg: Project config.
        persist: If True, save the index to `cfg.paths.vector_store_dir`.

    Returns:
        A `FAISS` vector store ready for retrieval.
    """
    embeddings = _get_embeddings(cfg)

    logger.info("Embedding %d chunks and building FAISS index...", len(chunks))
    vs = FAISS.from_documents(chunks, embeddings)

    if persist:
        cfg.paths.vector_store_dir.mkdir(parents=True, exist_ok=True)
        vs.save_local(str(cfg.paths.vector_store_dir))
        logger.info("FAISS index persisted to %s", cfg.paths.vector_store_dir)

    return vs


def load_vector_store(cfg: FineTuneConfig) -> FAISS:
    """
    Reload a previously-built FAISS index from disk.

    Raises:
        FileNotFoundError: If no index exists at `cfg.paths.vector_store_dir`.
    """
    index_dir = cfg.paths.vector_store_dir
    if not (index_dir / "index.faiss").exists():
        raise FileNotFoundError(
            f"No FAISS index at {index_dir}. "
            "Build one first by running build_vector_store()."
        )

    embeddings = _get_embeddings(cfg)
    logger.info("Loading FAISS index from %s", index_dir)
    # allow_dangerous_deserialization=True needed for newer langchain versions
    # since FAISS uses pickle; safe because we wrote it ourselves.
    return FAISS.load_local(
        str(index_dir), embeddings, allow_dangerous_deserialization=True
    )


# ===========================================================================
# 5. Retriever
# ===========================================================================
def build_retriever(vector_store: FAISS, cfg: FineTuneConfig):
    """
    Wrap a FAISS vector store in a LangChain retriever.

    Supports plain similarity search (default) and MMR (Maximal Marginal
    Relevance) for diversity-aware retrieval — useful when many chunks
    near the query are near-duplicates of each other.
    """
    if cfg.rag.search_type == "mmr":
        logger.info(
            "Building MMR retriever (k=%d, fetch_k=%d, lambda=%.2f)",
            cfg.rag.top_k,
            cfg.rag.mmr_fetch_k,
            cfg.rag.mmr_lambda,
        )
        return vector_store.as_retriever(
            search_type="mmr",
            search_kwargs={
                "k": cfg.rag.top_k,
                "fetch_k": cfg.rag.mmr_fetch_k,
                "lambda_mult": cfg.rag.mmr_lambda,
            },
        )

    logger.info("Building similarity retriever (k=%d)", cfg.rag.top_k)
    return vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={"k": cfg.rag.top_k},
    )


# ===========================================================================
# 6. RAG chain
# ===========================================================================
def build_rag_chain(llm, retriever, cfg: FineTuneConfig) -> RetrievalQA:
    """
    Wire the LLM + retriever + RAG prompt template into a LangChain RetrievalQA.

    Args:
        llm: A LangChain-compatible LLM (use `src.llm_loader.load_llm`).
        retriever: A LangChain retriever (use `build_retriever`).
        cfg: Project config (uses cfg.rag.rag_prompt_template).

    Returns:
        A `RetrievalQA` chain. Call `.invoke({"query": "..."})` to query.
    """
    prompt = PromptTemplate(
        template=cfg.rag.rag_prompt_template,
        input_variables=["context", "question"],
    )

    chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",            # "stuff" = concat all chunks into prompt
        retriever=retriever,
        chain_type_kwargs={"prompt": prompt},
        return_source_documents=True,  # critical for Streamlit display + RAGAS
    )
    logger.info("RAG chain ready (chain_type='stuff', return_sources=True)")
    return chain


# ===========================================================================
# 7. Inference helper
# ===========================================================================
def answer(chain: RetrievalQA, question: str) -> Dict[str, Any]:
    """
    Run a single RAG query end-to-end.

    Returns:
        {
            "question": str,
            "answer":   str,            # cleaned model output
            "sources":  List[Document]  # the top-k retrieved chunks
        }
    """
    result = chain.invoke({"query": question})
    # Different LangChain versions return slightly different shapes — normalize.
    raw_answer = result.get("result") or result.get("answer") or ""
    sources = result.get("source_documents") or []

    return {
        "question": question,
        "answer": raw_answer.strip(),
        "sources": sources,
    }


# ===========================================================================
# 8. End-to-end convenience builder
# ===========================================================================
def build_full_pipeline(
    cfg: Optional[FineTuneConfig] = None,
    *,
    llm_mode: str = "auto",
    rebuild_index: bool = False,
):
    """
    One-call helper: load corpus, build/load index, load LLM, return RAG chain.

    Returns the chain plus a few intermediate objects so callers can inspect
    the index or swap the LLM later.
    """
    cfg = cfg or get_default_config()

    # Vector store: build if missing, else load
    if rebuild_index or not (cfg.paths.vector_store_dir / "index.faiss").exists():
        docs = load_documents(cfg)
        chunks = chunk_documents(docs, cfg)
        vs = build_vector_store(chunks, cfg)
    else:
        vs = load_vector_store(cfg)

    retriever = build_retriever(vs, cfg)

    # LLM (imported lazily to keep this module importable without GPU)
    llm = load_llm(cfg, mode=llm_mode)

    chain = build_rag_chain(llm, retriever, cfg)
    return {
        "chain": chain,
        "retriever": retriever,
        "vector_store": vs,
        "llm": llm,
    }


__all__ = [
    "build_corpus_from_hf",
    "load_documents",
    "chunk_documents",
    "build_vector_store",
    "load_vector_store",
    "build_retriever",
    "build_rag_chain",
    "answer",
    "build_full_pipeline",
]

---
## Step 7 — Run Phase 2: build the FAISS index

Bootstraps a 1k-document corpus from the same Amazon QA dataset, chunks it,
embeds with `all-MiniLM-L6-v2`, writes a FAISS flat index.

In [ ]:
# 7.1 Bootstrap corpus + load
corpus_path = build_corpus_from_hf(cfg, output_format='csv', overwrite=False)
print('Corpus path:', corpus_path)

docs = load_documents(cfg)
print(f'Loaded {len(docs)} documents')
print('\nExample:\n', docs[0].page_content[:300])

In [ ]:
# 7.2 Chunk
chunks = chunk_documents(docs, cfg)
print(f'{len(docs)} docs → {len(chunks)} chunks')

In [ ]:
# 7.3 Build FAISS index (writes to artifacts/faiss_index/)
vs = build_vector_store(chunks, cfg, persist=True)
print(f'\nFAISS index built: {vs.index.ntotal} vectors, dim={vs.index.d}')

In [ ]:
# 7.4 Retrieval-only sanity check (no LLM)
retriever = build_retriever(vs, cfg)
hits = retriever.invoke('Is this Bluetooth speaker compatible with iPhone?')
print(f'Top-{len(hits)} chunks:\n')
for i, doc in enumerate(hits, 1):
    snippet = doc.page_content[:160].replace('\n', ' ')
    print(f'[{i}] {snippet}...')

---
## Step 8 — Three-way qualitative ablation

Same 3 questions through:
1. **Base LLM** (no RAG, no fine-tuning)
2. **RAG + Base** (retrieval-augmented, but no fine-tuning)
3. **RAG + Fine-tuned** (the full system)

Phase 3 will replace this eyeball test with RAGAS quantitative scoring.

In [ ]:
import gc, torch

QUERIES = [
    'Is the Echo Dot 5th gen waterproof?',
    'Does this USB-C cable support 100W fast charging?',
    'How long is the warranty on Kindle Paperwhite?',
]
results = {q: {} for q in QUERIES}

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# 8a — Base LLM only
base_llm = load_llm(cfg, mode='base')
for q in QUERIES:
    prompt = cfg.prompt_template.format(question=q, answer='').rstrip()
    out = base_llm.invoke(prompt)
    results[q]['base'] = out.strip()[:250]
    print(f'\nQ: {q}\nA: {results[q]["base"]}')
del base_llm; free_gpu()

In [ ]:
# 8b — RAG + Base LLM
base_llm = load_llm(cfg, mode='base')
rag_base = build_rag_chain(base_llm, retriever, cfg)
for q in QUERIES:
    out = answer(rag_base, q)
    results[q]['rag_base'] = out['answer'][:250]
    print(f'\nQ: {q}\nA: {results[q]["rag_base"]}')
del base_llm, rag_base; free_gpu()

In [ ]:
# 8c — RAG + Fine-tuned LLM (the full system)
ft_llm = load_llm(cfg, mode='finetuned')
rag_ft = build_rag_chain(ft_llm, retriever, cfg)
for q in QUERIES:
    out = answer(rag_ft, q)
    results[q]['rag_ft'] = out['answer'][:250]
    print(f'\nQ: {q}\nA: {results[q]["rag_ft"]}')

In [ ]:
# Pretty side-by-side
import pandas as pd
df = pd.DataFrame(results).T
df.columns = ['Base LLM', 'RAG + Base', 'RAG + Fine-tuned']
df.index.name = 'Question'
df

---
## Step 9 — Persist artifacts to Google Drive (optional)

Colab sessions die after ~12h idle. Backing up the LoRA adapter + FAISS
index to Drive means later phases can resume from a fresh runtime.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DEST = '/content/drive/MyDrive/AmazonQA-RAG'
    !mkdir -p "{DEST}/lora_adapter" "{DEST}/faiss_index"
    !cp -r {cfg.paths.adapter_dir}/* "{DEST}/lora_adapter/"
    !cp -r {cfg.paths.vector_store_dir}/* "{DEST}/faiss_index/"
    print(f'Artifacts backed up to: {DEST}')
except Exception as e:
    print(f'(Skipping Drive backup: {e})')

---
## Smoke test complete

If every cell ran without error and Step 8 produced sensible answers,
**both Phase 1 (QLoRA fine-tuning) and Phase 2 (RAG pipeline) are working**.

### Going from smoke test → production run

Re-run after deleting these smoke-test overrides in the Step 2 config cell:

```python
cfg.dataset.train_sample_size = 500    # remove → uses 10,000
cfg.training.max_steps = 50            # remove → trains 1 full epoch
cfg.rag.corpus_sample_size = 1000      # remove → indexes 5,000 docs
```

### Troubleshooting

| Symptom | Fix |
|---|---|
| `Your session crashed` after running cell 4.2 (model load) | OOM. Set `USE_PHI2 = True` in Step 1 and re-run from Step 1 onward. Mistral-7B is on the edge of T4 free-tier. |
| `numpy.dtype size changed, may indicate binary incompatibility` | NumPy 1.x didn't load. Verify with `import numpy; numpy.__version__` — if it shows 2.x, do **Runtime -> Disconnect and delete runtime** and start over from Step 0. |
| `cannot import name 'HF_HUB_ENABLE_HF_TRANSFER'` | huggingface_hub version mismatch. Re-run Step 1's atomic install cell — version 0.25.2 is the correct lock. |
| `NameError: load_and_preprocess_dataset` after a kernel restart | Memory was wiped. Re-run Step 2 (config), the cfg cell, and Step 3 (finetune) cells in order. Packages persist on disk; function definitions don't. |
| `OSError: ... Mistral gated` | Accept license at https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2 + add `HF_TOKEN` Colab secret. Or just leave `USE_PHI2 = True`. |
| LLM repeats the prompt verbatim | `return_full_text=False` is set inside `load_llm` — verify Step 5's cell ran (look for the `def load_llm` definition in `dir()`). |
| `FAISS deserialization` error | `allow_dangerous_deserialization=True` is already set in `load_vector_store` — should not happen. If it does, re-run Step 6 (RAG inline). |

### Critical lesson on kernel restarts

Whenever Colab restarts the kernel (deliberately via Step 1's restart cell, or after an OOM crash):
- **Disk-installed packages are preserved**
- **In-memory variables and function definitions are wiped**

After ANY restart, you must re-run the inlined module cells (Step 2, Step 3, Step 5, Step 6) before any cell that calls those functions. Skip the install cell and the kill cell.

### Next: Phase 3 — RAGAS evaluation

Will quantify what Step 8 showed qualitatively, using: `faithfulness`, `answer_relevancy`, `context_precision`, `context_recall`.